### サポートベクトルマシン(SVMとは) 
- クラス間のマージンを最大化する決定境界を求める手法
- 「マージン」とは、決定境界から最も近いデータ点までの距離のこと。このマージンを最大にすることで、未知データに対して頑健な分類を作る。
- 決定境界に最も近いデータ点をサポートベクトルと呼び、これだけが境界の決定に影響する。

||内容|
|----|----|
|目的|マージンの最大化（∣w∣ の最小化）|
|損失 関数|ヒンジ損失|
|正則化|C でマージンと誤分類のトレードオフを制御|
|最適化|勾配降下法|


In [1]:
# ライブラリでの実装
# 線形SVMの導出を分かりやすくするため、今回は品種0（setosa）と品種1（versicolor）の2クラスを対象とする。

import numpy as np
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X = iris.data[iris.target != 2]
y = iris.target[iris.target != 2]
y = np.where(y == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = SVC(kernel="linear", C=1.0)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")

正解率: 1.0000


In [2]:
# scratchでのSVMの実装


import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


class SVM:
    def __init__(self, lr=0.001, C=1.0, n_epochs=1000):
        self.lr = lr
        self.C = C
        self.n_epochs = n_epochs
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for _ in range(self.n_epochs):
            for i in range(n_samples):
                condition = y[i] * (X[i] @ self.w + self.b) >= 1
                if condition:
                    self.w -= self.lr * self.w
                else:
                    self.w -= self.lr * (self.w - self.C * y[i] * X[i])
                    self.b += self.lr * self.C * y[i]

    def predict(self, X):
        return np.sign(X @ self.w + self.b).astype(int)

iris = load_iris()
X = iris.data[iris.target != 2]
y = iris.target[iris.target != 2]
y = np.where(y == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = SVM(lr=0.001, C=1.0, n_epochs=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")


正解率: 1.0000


ラベルの符号

`y = np.where(y == 0, -1, 1)` でラベルを {−1,1} に変換しています。これは、`condition` の計算で $ y_i(w^Tx_i+b)$の符号が正しく機能するようにするためです。

`fit` メソッドの更新式

導出した勾配をそのまま実装しています。制約を満たすとき（`condition=True`）は正則化項 w のみで更新し、満たさないとき（`condition=False`）はヒンジ損失の勾配も加えて更新します。

`predict` メソッド

w^T*x+b の符号で分類します。`np.sign` は正なら1、負なら-1を返します。